## Finding Dataset

In order to realize the study, datasets representing the crimes were searched for, with the City of Chicago being considered the best. It offered an easy api to obtain data as a detailed dataset displaying crimes per day, which was of great value for the project. The "$limit=5000000" filter was added in order to be able to download all entries, as the API seems to have a default limit a little over a thousand, and the crude dataset goes beyond a million.

Before continuing, the dataset was downloaded to a file for the manual read of the data.

In [20]:
import pandas as pd

url = {
        2025: "https://svs.gsfc.nasa.gov/vis/a000000/a005400/a005415/mooninfo_2025.json",
        2024: "https://svs.gsfc.nasa.gov/vis/a000000/a005100/a005187/mooninfo_2024.json",
        2023: "https://svs.gsfc.nasa.gov/vis/a000000/a005000/a005048/mooninfo_2023.json",
        2022: "https://svs.gsfc.nasa.gov/vis/a000000/a004900/a004955/mooninfo_2022.json",
        2021: "https://svs.gsfc.nasa.gov/vis/a000000/a004800/a004874/mooninfo_2021.json",
        2020: "https://svs.gsfc.nasa.gov/vis/a000000/a004700/a004768/mooninfo_2020.json", 
}

frames = []

for y, u in url.items():
    print(f"Downloading {y}")
    try:
        df_year = pd.read_json(u)
        df_year["year"] = y
        frames.append(df_year)
    except Exception as e:
        print(f"Failed to download year {y}")

df = pd.concat(frames, ignore_index=True)

print("Downloaded Moon phases datasets over 2020", len(df), "records")
print(df.columns)


Downloaded Moon phases datasets over 2020 52610 records
Index(['time', 'phase', 'age', 'diameter', 'distance', 'j2000', 'subsolar',
       'subearth', 'posangle', 'year'],
      dtype='object')


Out of the columns of the dataset, the ones considered important were: 

- time
- year
- age
- phase
- diameter
- distance


In [23]:
df["time"] = pd.to_datetime(df["time"], errors="coerce")

df["time"] = df["time"].dt.normalize() 

daily = (
    df.sort_values("time")          
      .groupby("time", as_index=False)
      .agg({
          "time": "first",
          "year": "first",
          "age": "first",
          "phase": "first",
          "diameter": "first",
          "distance": "first"
      })
)

daily.to_csv("moon_phases_clean.csv", index=False)

print(len(daily), "rows in grouped dataset")

print(daily.head())
print(daily.columns)
print(daily["age"].unique())



2192 rows in grouped dataset
        time  year     age  phase  diameter  distance
0 2020-01-01  2020   6.491  36.26    1771.7    404538
1 2020-01-02  2020   7.491  45.51    1772.6    404346
2 2020-01-03  2020   8.366  53.78    1778.0    403098
3 2020-01-04  2020   9.699  66.27    1794.7    399349
4 2020-01-05  2020  10.366  72.27    1806.5    396742
Index(['time', 'year', 'age', 'phase', 'diameter', 'distance'], dtype='object')
[ 6.491  7.491  8.366 ...  9.553 10.887 11.512]


In [1]:
def moon_phase_from_age(df):
    
    df.loc[
        (df["age"] >= 0) & (df["age"] < 1.845),
        "phase_str"] = "New Moon"

    df.loc[
        (df["age"] >= 1.845) & (df["age"] < 5.536),
        "phase_str"] = "Waxing Crescent"

    df.loc[
        (df["age"] >= 5.536) & (df["age"] < 9.228),
        "phase_str"] = "First Quarter"

    df.loc[
        (df["age"] >= 9.228) & (df["age"] < 12.919), 
        "phase_str"] = "Waxing Gibbous"

    df.loc[
        (df["age"] >= 12.919) & (df["age"] < 16.610),
        "phase_str"] = "Full Moon"

    df.loc[
        (df["age"] >= 16.610) & (df["age"] < 20.302),
        "phase_str"] = "Waning Gibbous"

    df.loc[
        (df["age"] >= 20.302) & (df["age"] < 23.993), 
        "phase_str"] = "Last Quarter"

    df.loc[
        (df["age"] >= 23.993) & (df["age"] <= 29.53), 
        "phase_str"] = "Waning Crescent"
